In [27]:
import math
import os

import rclpy
from ament_index_python.packages import get_package_share_directory
from gazebo_msgs.srv import DeleteEntity, SpawnEntity
from rclpy.node import Node

In [28]:
def _load_waffle_pi_sdf_xml() -> str:
    """读取 waffle_pi 小车的 SDF 模型文件。"""
    model_folder = "turtlebot3_waffle_pi"
    search_paths = []

    try:
        share_dir = get_package_share_directory("turtlebot3_gazebo")
        search_paths.append(os.path.join(share_dir, "models", model_folder, "model.sdf"))
    except Exception:
        pass

    for base_path in os.environ.get("GAZEBO_MODEL_PATH", "").split(":"):
        if base_path.strip():
            search_paths.append(os.path.join(base_path.strip(), model_folder, "model.sdf"))

    for candidate in search_paths:
        if os.path.exists(candidate):
            with open(candidate, "r", encoding="utf-8") as file:
                return file.read()

    raise FileNotFoundError(
        "未找到 turtlebot3_waffle_pi/model.sdf。请确认已安装 turtlebot3_gazebo，"
        "并且 Gazebo 模型路径可用。"
    )


def _delete_model_if_exists(model_name: str) -> None:
    """如果 Gazebo 中已有同名模型，则先尝试删除。"""
    node = Node(f"delete_model_if_exists_{model_name}")
    try:
        delete_cli = node.create_client(DeleteEntity, "/delete_entity")
        if not delete_cli.wait_for_service(timeout_sec=5.0):
            raise RuntimeError("/delete_entity service not available. Is Gazebo running?")

        req = DeleteEntity.Request()
        req.name = model_name
        fut = delete_cli.call_async(req)
        rclpy.spin_until_future_complete(node, fut, timeout_sec=10.0)
    finally:
        node.destroy_node()

In [29]:
def spawn_waffle_pi(
    x: float,
    y: float,
    yaw: float = 0.0,
    z: float = 0.01,
    name: str = "waffle_pi",
    delete_if_exists: bool = True,
 ) -> tuple[float, float, float]:
    """在 Gazebo 中生成一个 waffle_pi 小车，并返回使用的 (x, y, yaw)。"""
    owns_rclpy = not rclpy.ok()
    if owns_rclpy:
        rclpy.init(args=None)

    try:
        if delete_if_exists:
            _delete_model_if_exists(name)

        node = Node(f"spawn_waffle_pi_node_{name}")
        try:
            spawn_cli = node.create_client(SpawnEntity, "/spawn_entity")
            if not spawn_cli.wait_for_service(timeout_sec=10.0):
                raise RuntimeError("/spawn_entity service not available. Is Gazebo running?")

            req = SpawnEntity.Request()
            req.name = name
            req.xml = _load_waffle_pi_sdf_xml()
            req.initial_pose.position.x = float(x)
            req.initial_pose.position.y = float(y)
            req.initial_pose.position.z = float(z)
            req.initial_pose.orientation.z = math.sin(float(yaw) / 2.0)
            req.initial_pose.orientation.w = math.cos(float(yaw) / 2.0)

            fut = spawn_cli.call_async(req)
            rclpy.spin_until_future_complete(node, fut, timeout_sec=15.0)
            response = fut.result()
            if response is None:
                raise RuntimeError("SpawnEntity 调用失败")
            if hasattr(response, "success") and not response.success:
                raise RuntimeError(f"生成 {name} 失败: {response.status_message}")

            return float(x), float(y), float(yaw)
        finally:
            node.destroy_node()
    finally:
        if owns_rclpy:
            rclpy.shutdown()

In [30]:
import numpy as np
Pi = np.pi
robot_pose = spawn_waffle_pi(x=0.0, y=0.0, yaw=-Pi/2, name="waffle_pi")
print("spawned:", robot_pose)


spawned: (0.0, 0.0, -1.5707963267948966)


In [31]:
from sensor_msgs.msg import LaserScan


def inspect_lidar_direction(topic_name: str = "/scan", timeout_sec: float = 5.0):
    """接收一帧 LaserScan，并判断角度是顺时针还是逆时针增加。"""
    owns_rclpy = not rclpy.ok()
    if owns_rclpy:
        rclpy.init(args=None)

    class ScanProbe(Node):
        def __init__(self):
            super().__init__("inspect_lidar_direction_node")
            self.msg = None
            self.sub = self.create_subscription(LaserScan, topic_name, self._on_scan, 10)

        def _on_scan(self, msg: LaserScan):
            self.msg = msg

    node = ScanProbe()
    try:
        deadline_ns = node.get_clock().now().nanoseconds + int(timeout_sec * 1e9)
        while node.msg is None and node.get_clock().now().nanoseconds < deadline_ns:
            rclpy.spin_once(node, timeout_sec=0.1)

        if node.msg is None:
            print(f"在 {timeout_sec:.1f}s 内没有收到 {topic_name} 的 LaserScan 消息")
            print("请确认 Gazebo 正在运行，且雷达话题名正确。")
            return None

        msg = node.msg
        angle_min = float(msg.angle_min)
        angle_max = float(msg.angle_max)
        angle_increment = float(msg.angle_increment)
        sample_count = len(msg.ranges)

        if angle_increment > 0.0:
            direction = "逆时针"
            detail = "索引增大时，扫描角度从右侧负角逐步转到左侧正角"
        elif angle_increment < 0.0:
            direction = "顺时针"
            detail = "索引增大时，扫描角度从左侧正角逐步转到右侧负角"
        else:
            direction = "无法判断"
            detail = "angle_increment == 0，数据不符合正常 LaserScan 约定"

        print(f"topic: {topic_name}")
        print(f"sample_count: {sample_count}")
        print(f"angle_min: {angle_min:.6f} rad")
        print(f"angle_max: {angle_max:.6f} rad")
        print(f"angle_increment: {angle_increment:.6f} rad")
        print(f"方向判断: {direction}")
        print(detail)

        return {
            "topic": topic_name,
            "sample_count": sample_count,
            "angle_min": angle_min,
            "angle_max": angle_max,
            "angle_increment": angle_increment,
            "direction": direction,
            "detail": detail,
        }
    finally:
        node.destroy_node()
        if owns_rclpy:
            rclpy.shutdown()


lidar_info = inspect_lidar_direction()
lidar_info

topic: /scan
sample_count: 360
angle_min: 0.000000 rad
angle_max: 6.280000 rad
angle_increment: 0.017493 rad
方向判断: 逆时针
索引增大时，扫描角度从右侧负角逐步转到左侧正角


{'topic': '/scan',
 'sample_count': 360,
 'angle_min': 0.0,
 'angle_max': 6.28000020980835,
 'angle_increment': 0.01749303564429283,
 'direction': '逆时针',
 'detail': '索引增大时，扫描角度从右侧负角逐步转到左侧正角'}